# Building an Agent Loop with Claude — C# (Polyglot Notebook)

This is the **C# equivalent** of `agent_loop.ipynb`, implemented using the [Anthropic.SDK](https://www.nuget.org/packages/Anthropic.SDK) NuGet package.

---

### What is an Agent Loop?
An agent loop is a multi-turn conversation cycle where Claude can:
1. Pause generation and request an external tool call,
2. Receive the tool result,
3. Continue reasoning until it produces a final answer.

```text
  ┌─────────────────┐
  │   User Prompt   │
  └────────┬────────┘
           │
           ▼
  ┌────────────────────────────────────────────────────────┐
  │  Call Claude Messages API (with tools & conversation)  │◄────────┐
  └────────────────────────┬───────────────────────────────┘         │
                           │                                         │
                           ▼                                         │
                 [ Inspect StopReason ]                              │
                           │                                         │
            ┌──────────────┴──────────────┐                          │
       "tool_use"                    "end_turn"                      │
            │                             │                          │
            ▼                             ▼                          │
  ┌────────────────────┐        ┌──────────────────┐                 │
  │  Execute C# method │        │  Print & return  │                 │
  │     (RunTool)      │        │   final answer   │                 │
  └─────────┬──────────┘        └──────────────────┘                 │
            │                                                        │
            ▼                                                        │
  ┌───────────────────────────────────────────────┐                  │
  │ Append tool_result to messages conversation   │──────────────────┘
  └───────────────────────────────────────────────┘
```

> ⚠️ **Credit Note:** Do not execute API request cells unless you are ready to consume tokens.

## Step 1: Install the Anthropic NuGet Package

Polyglot Notebooks lets you install NuGet packages inline using `#r "nuget:..."`. This installs `Anthropic.SDK` and its dependencies directly into the notebook session.

> **Note:** This only needs to be run once per session. You do **not** need to edit a `.csproj` file.

In [ ]:
#r "nuget: Anthropic.SDK, *-*"
#r "nuget: Microsoft.Extensions.Configuration, *-*"
#r "nuget: Microsoft.Extensions.Configuration.EnvironmentVariables, *-*"
#r "nuget: dotenv.net, *-*"

Console.WriteLine("✅ Packages loaded successfully.");

## Step 2: Load API Key from `.env`

We use `dotenv.net` to load the `.env` file from the project root. The library searches parent directories automatically with `DotEnv.Fluent().WithProbeForEnv(probeLevelsToSearch: 5).Load()`.

Your `.env` file should contain:
```env
ANTHROPIC_API_KEY=your-api-key-here
```

In [ ]:
using dotenv.net;

// Search up to 5 parent directories for .env
DotEnv.Fluent()
    .WithProbeForEnv(probeLevelsToSearch: 5)
    .Load();

// Prefer ANTHROPIC_API_KEY, fall back to API_KEY
var apiKey = Environment.GetEnvironmentVariable("ANTHROPIC_API_KEY")
          ?? Environment.GetEnvironmentVariable("API_KEY");

if (string.IsNullOrEmpty(apiKey))
    throw new InvalidOperationException(
        "API Key not found. Ensure ANTHROPIC_API_KEY is set in your .env file.");

Console.WriteLine("✅ API Key loaded from environment.");

## Step 3: Initialize the Anthropic Client

The `AnthropicClient` from `Anthropic.SDK` is the main entry point. Pass your API key via `AnthropicClient(apiKey)`. Alternatively, set `ANTHROPIC_API_KEY` in your environment and call `new AnthropicClient()` without arguments.

In [ ]:
using Anthropic.SDK;
using Anthropic.SDK.Models;
using Anthropic.SDK.Messaging;

var client = new AnthropicClient(apiKey);
Console.WriteLine("✅ AnthropicClient initialized.");

## Step 4: Define the Tool Schema

Tools are declared using `Tool.FromFunc<>()` or manually constructed `Tool` objects.

Each tool needs:
- **`Name`**: Unique identifier for Claude to reference.
- **`Description`**: Descriptive text Claude uses to decide whether to call the tool.
- **`InputSchema`**: JSON Schema defining expected input properties.

> **Tip:** Write clear, specific descriptions. Claude selects tools based primarily on how well the description matches the user's intent.

In [ ]:
using System.Text.Json;
using System.Text.Json.Nodes;

// Build JSON schema for the get_weather tool input
var weatherInputSchema = new JsonObject
{
    ["type"] = "object",
    ["properties"] = new JsonObject
    {
        ["city"] = new JsonObject
        {
            ["type"] = "string",
            ["description"] = "The name of the city to get the weather for, e.g. Austin, TX"
        }
    },
    ["required"] = new JsonArray { "city" }
};

// Declare tools list
var tools = new List<Anthropic.SDK.Messaging.Tool>
{
    new Anthropic.SDK.Messaging.Tool
    {
        Name        = "get_weather",
        Description = "Get the current weather and temperature for a given city.",
        InputSchema = weatherInputSchema
    }
};

Console.WriteLine($"✅ {tools.Count} tool(s) registered.");

## Step 5: Implement the Tool Dispatcher (`RunTool`)

When Claude returns `StopReason == "tool_use"`, we parse the `ToolUseBlock` and dispatch to the right local method.

- `block.Name` — which tool Claude chose.
- `block.Input` — a `JsonElement` containing the arguments matching the tool's schema.

In a real application, this is where you would call an external REST API, database query, or any business logic.

In [ ]:
string RunTool(string name, JsonElement input)
{
    if (name == "get_weather")
    {
        var city = input.GetProperty("city").GetString() ?? "Unknown";
        // In production: call a real weather API (e.g. OpenWeatherMap)
        return $"Weather in {city}: 95°F, Sunny with light winds.";
    }

    throw new NotImplementedException($"Unknown tool requested: {name}");
}

Console.WriteLine("✅ Tool dispatcher (RunTool) defined.");

## Step 6: Build the Initial Conversation

The `messages` list holds the full conversation history, exactly mirroring the Python SDK:
- `role: "user"` — user prompts and tool results.
- `role: "assistant"` — Claude's responses (text and tool requests).

In [ ]:
var messages = new List<Message>
{
    new Message
    {
        Role    = RoleType.User,
        Content = new List<ContentBase>
        {
            new TextContent { Text = "What should I wear in Austin today?" }
        }
    }
};

Console.WriteLine("✅ Initial conversation initialized.");

## Step 7: The Agent Execution Loop

Each iteration of the loop sends the full message history to Claude and inspects `response.StopReason`:

| `StopReason` | Meaning | Action |
| :--- | :--- | :--- |
| `"end_turn"` | Claude produced a final answer | Print response text, exit loop |
| `"tool_use"` | Claude needs external data | Execute tool, append result to messages, loop again |
| `"max_tokens"` | Response hit token limit | Handle truncation or increase `MaxTokens` |

**Key Message Format for Tool Results:**
```csharp
// 1. Append assistant's response (which includes tool requests)
messages.Add(new Message { Role = RoleType.Assistant, Content = response.Content });

// 2. Append your tool results as a user message
messages.Add(new Message {
    Role = RoleType.User,
    Content = new List<ContentBase> {
        new ToolResultContent {
            ToolUseId = block.Id,
            Content   = toolResult
        }
    }
});
```

In [ ]:
// ⚠️ Running this cell sends live requests to the Anthropic API and consumes tokens.

var parameters = new MessageParameters
{
    Model     = AnthropicModels.Claude45Haiku,   // claude-sonnet-5-5 or equivalent
    MaxTokens = 1024,
    Tools     = tools,
    Messages  = messages
};

int maxIterations    = 5;
int currentIteration = 0;

while (currentIteration < maxIterations)
{
    currentIteration++;
    Console.WriteLine($"\n--- Iteration {currentIteration} ---");

    // Update the messages in the parameters each turn
    parameters.Messages = messages;

    var response = await client.Messages.GetClaudeMessageAsync(parameters);

    // Case A: Claude finished — print final answer and exit
    if (response.StopReason == "end_turn")
    {
        Console.WriteLine("\n[Final Answer from Claude]:");
        foreach (var block in response.Content)
        {
            if (block is TextContent textBlock)
                Console.WriteLine(textBlock.Text);
        }
        break;
    }

    // Case B: Claude requested tool execution
    if (response.StopReason == "tool_use")
    {
        var toolResults = new List<ContentBase>();

        foreach (var block in response.Content)
        {
            if (block is TextContent intermediateText)
                Console.WriteLine($"[Claude thought]: {intermediateText.Text}");

            if (block is ToolUseContent toolUseBlock)
            {
                Console.WriteLine($"[Tool Call]: {toolUseBlock.Name} => {toolUseBlock.Input}");

                var toolResult = RunTool(toolUseBlock.Name, toolUseBlock.Input);
                Console.WriteLine($"[Tool Result]: {toolResult}");

                toolResults.Add(new ToolResultContent
                {
                    ToolUseId = toolUseBlock.Id,
                    Content   = toolResult
                });
            }
        }

        // Feed Claude's response and tool results back into the conversation
        messages.Add(new Message { Role = RoleType.Assistant, Content = response.Content });
        messages.Add(new Message { Role = RoleType.User,      Content = toolResults });
    }
    else
    {
        Console.WriteLine($"Loop terminated with stop reason: {response.StopReason}");
        break;
    }
}

## Step 8: Key Takeaways (C# vs Python Comparison)

| Concept | Python SDK | C# SDK (`Anthropic.SDK`) |
| :--- | :--- | :--- |
| Client init | `anthropic.Anthropic(api_key=...)` | `new AnthropicClient(apiKey)` |
| Send message | `client.messages.create(...)` | `await client.Messages.GetClaudeMessageAsync(...)` |
| Stop reason check | `response.stop_reason == "tool_use"` | `response.StopReason == "tool_use"` |
| Text block | `block.type == "text"` → `block.text` | `block is TextContent tb` → `tb.Text` |
| Tool use block | `block.type == "tool_use"` → `block.name, block.input` | `block is ToolUseContent tu` → `tu.Name, tu.Input` |
| Tool result | Dict with `type`, `tool_use_id`, `content` | `new ToolResultContent { ToolUseId, Content }` |
| Async | Synchronous by default | Fully async — use `await` |

---

**Remember:**
- Always append **both** the assistant's turn (with tool requests) and the user's `ToolResultContent` before looping.
- Always cap iterations with `maxIterations` to prevent runaway loops.
- Parallel tool calls: Claude may return multiple `ToolUseContent` blocks in a single response — iterate through all of them.